In [1]:
import pandas as pd

In [ ]:
df = pd.read_excel('/Users/maria/Desktop/Code/HSE/TS/data (1).xls')
df

,Индексы потребительских цен на товары и услуги (процент),Unnamed: 1,Unnamed: 2,Unnamed: 3
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,Город Москва столица Российской Федерации горо...
2,NaN,NaN,NaN,"Кофе в организациях быстрого обслуживания, 200 г"
3,2002.0,январь,К предыдущему месяцу,102.57
4,2002.0,февраль,К предыдущему месяцу,100.09
...,...,...,...,...
283,2025.0,август,К предыдущему месяцу,100.61
284,2025.0,сентябрь,К предыдущему месяцу,101.05
285,2025.0,октябрь,К предыдущему месяцу,100.88
286,2025.0,ноябрь,К предыдущему месяцу,100.13


попросила нейронку преобразовать датасет в более удобный

In [9]:
df = df.rename(columns={
    df.columns[0]: "year",
    df.columns[1]: "month",
    df.columns[2]: "calc",
    df.columns[3]: "value",
})

region = df.loc[1, "value"] if 1 in df.index else None
product = df.loc[2, "value"] if 2 in df.index else None

meta = {"region": region, "product": product}

df = df.dropna(subset=["year", "month", "calc", "value"]).copy()

df["calc"] = df["calc"].astype(str).str.strip()
df = df[df["calc"].eq("К предыдущему месяцу")].copy()

month_map = {
    "январь": 1, "февраль": 2, "март": 3, "апрель": 4, "май": 5, "июнь": 6,
    "июль": 7, "август": 8, "сентябрь": 9, "октябрь": 10, "ноябрь": 11, "декабрь": 12,
}

df["month"] = df["month"].astype(str).str.strip().str.lower()
df["month_num"] = df["month"].map(month_map)

df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
df["value"] = pd.to_numeric(df["value"], errors="coerce")

df = df.dropna(subset=["year", "month_num", "value"]).copy()

df["date"] = pd.to_datetime(
    dict(year=df["year"].astype(int), month=df["month_num"].astype(int), day=1)
)

series = df[["date", "value"]].sort_values("date").reset_index(drop=True)

print(meta)
series



{'region': None, 'product': None}


,date,value
0,2002-01-01,102.57
1,2002-02-01,100.09
2,2002-03-01,103.36
3,2002-04-01,100.00
4,2002-05-01,101.10
...,...,...
280,2025-08-01,100.61
281,2025-09-01,101.05
282,2025-10-01,100.88
283,2025-11-01,100.13


In [10]:
from hw4_calc import fit_growth_curves, print_summaries, save_step1_plots

compare_tbl, diag_tbl, models, fitted = fit_growth_curves(series)

compare_tbl


,model,n,k,R2,Adj_R2,AIC,BIC,RMSE,MAE
0,Log-linear: ln(y) ~ t,285,2,0.047287,0.043921,-1718.280291,-1710.975313,0.011790,0.007316
1,Quadratic: y ~ t + t^2,285,3,0.061850,0.055196,922.423874,933.381342,1.207827,0.722804
2,Linear: y ~ t,285,2,0.044735,0.041360,925.576231,932.881209,1.218794,0.743739


In [11]:
diag_tbl


,model,DW,LB(12)_stat,LB(12)_p,BP_stat,BP_p,JB_stat,JB_p
0,Linear: y ~ t,1.654761,19.517804,0.076775,0.220317,0.638799,19158.615485,0.0
1,Quadratic: y ~ t + t^2,1.684855,17.414440,0.134661,0.788236,0.674275,19536.526120,0.0
2,Log-linear: ln(y) ~ t,1.645518,20.776073,0.053757,0.177745,0.673318,15349.606776,0.0


In [12]:
print_summaries(models)



Linear: y ~ t
                            OLS Regression Results                            
Dep. Variable:                  value   R-squared:                       0.045
Model:                            OLS   Adj. R-squared:                  0.041
Method:                 Least Squares   F-statistic:                     13.25
Date:                Thu, 19 Feb 2026   Prob (F-statistic):           0.000323
Time:                        20:30:31   Log-Likelihood:                -460.79
No. Observations:                 285   AIC:                             925.6
Df Residuals:                     283   BIC:                             932.9
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        101.1640      0.145    6

In [13]:
save_step1_plots(fitted, models)


In [17]:
import importlib

import hw4_calc as hw4_calc

importlib.reload(hw4_calc)

<module 'hw4_calc' from '/Users/maria/Desktop/Code/HSE/TS/hw4_calc.py'>

In [16]:
from hw4_calc import chow_test_many, default_candidate_breaks

# series: DataFrame with columns ["date", "value"]
candidates = default_candidate_breaks()

# 1) Chow по линейному тренду
tbl_linear = chow_test_many(series, candidates, spec="linear", min_segment=24)
display(tbl_linear)

# 2) Chow по квадратичному тренду (как robustness check)
tbl_quad = chow_test_many(series, candidates, spec="quadratic", min_segment=24)
display(tbl_quad)


,break_date,spec,n_left,n_right,k,F,p_value,decision_5%
0,2020-04-01,linear,218,67,2,5.859446,0.003213,reject H0 (break)
1,2014-12-01,linear,156,129,2,3.881285,0.021739,reject H0 (break)
2,2022-03-01,linear,241,44,2,2.946655,0.054140,fail to reject
3,2008-09-01,linear,81,204,2,1.112796,0.330083,fail to reject


,break_date,spec,n_left,n_right,k,F,p_value,decision_5%
0,2020-04-01,quadratic,218,67,3,3.203713,0.023694,reject H0 (break)
1,2014-12-01,quadratic,156,129,3,2.026741,0.110368,fail to reject
2,2008-09-01,quadratic,81,204,3,1.088113,0.354494,fail to reject
3,2022-03-01,quadratic,241,44,3,0.443911,0.721799,fail to reject


In [18]:
from hw4_calc import fit_dummy_models, extract_key_coefs, save_step3_plots

# Берём точку сдвига из шага 2: 2020-04-01 (самая устойчивая)
break_date = "2020-04-01"

# 1) Линейный тренд + dummy (уровень / уровень+наклон)
cmp_lin, diag_lin, models_lin = fit_dummy_models(series, break_date, spec="linear", lb_lags=12)
display(cmp_lin)
display(diag_lin)

# 2) Коэффициенты модели сдвига (обычно показывают level+trend)
best_name = cmp_lin.loc[0, "model"]  # по AIC/BIC
best_res = models_lin[best_name].model
display(extract_key_coefs(best_res))

# 3) Сохранить графики (для вставки в pdf)
save_step3_plots(models_lin, out_dir="plots_step3_linear")

# (Опционально) robustness: квадратичный тренд + dummy
cmp_q, diag_q, models_q = fit_dummy_models(series, break_date, spec="quadratic", lb_lags=12)
display(cmp_q)
display(diag_q)


,model,n,k,R2,Adj_R2,AIC,BIC,RMSE,MAE
0,Dummy level+trend shift @ 2020-04-01 (linear),285,4,0.082742,0.072949,918.005254,932.615211,1.194302,0.698205
1,Dummy level shift @ 2020-04-01 (linear),285,3,0.070414,0.063821,919.810326,930.767794,1.202301,0.725174
2,Baseline (linear trend),285,2,0.044735,0.041360,925.576231,932.881209,1.218794,0.743739


,model,DW,LB(12)_stat,LB(12)_p,BP_stat,BP_p,JB_stat,JB_p
0,Baseline (linear trend),1.654761,19.517804,0.076775,0.220317,0.638799,19158.615485,0.0
1,Dummy level shift @ 2020-04-01 (linear),1.701630,17.240578,0.140769,3.064728,0.216024,17961.499990,0.0
2,Dummy level+trend shift @ 2020-04-01 (linear),1.723381,17.148469,0.144097,3.496475,0.321220,20805.043598,0.0


,coef,std_err,t,p_value
const,101.426016,0.163865,618.960672,0.000000
t,-0.006302,0.001303,-4.834737,0.000002
D,-2.930153,1.881772,-1.557124,0.120567
tD,0.014662,0.007545,1.943405,0.052966


,model,n,k,R2,Adj_R2,AIC,BIC,RMSE,MAE
0,Dummy level+trend shift @ 2020-04-01 (quadratic),285,5,0.088051,0.075023,918.350869,936.613315,1.190841,0.695947
1,Dummy level shift @ 2020-04-01 (quadratic),285,4,0.070755,0.060834,921.705755,936.315712,1.202081,0.723367
2,Baseline (quadratic trend),285,3,0.061850,0.055196,922.423874,933.381342,1.207827,0.722804


,model,DW,LB(12)_stat,LB(12)_p,BP_stat,BP_p,JB_stat,JB_p
0,Baseline (quadratic trend),1.684855,17.414440,0.134661,0.788236,0.674275,19536.526120,0.0
1,Dummy level shift @ 2020-04-01 (quadratic),1.701960,17.157750,0.143759,3.372946,0.337618,18185.749360,0.0
2,Dummy level+trend shift @ 2020-04-01 (quadratic),1.733835,17.671444,0.126037,3.542282,0.471478,21238.086157,0.0
